In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df=pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/master/data.csv")
df.head()

In [ ]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [ ]:
df.head()

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['diagnosis']),df['diagnosis'],test_size=0.2,random_state=42)

In [ ]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
encoder=LabelEncoder()
y_train= encoder.fit_transform(y_train)
y_test= encoder.fit_transform(y_test)


In [ ]:
y_train

In [ ]:
X_train_tensor= torch.from_numpy(X_train.astype(np.float32))
X_test_tensor= torch.from_numpy(X_test.astype(np.float32))
y_train_tensor= torch.from_numpy(y_train.astype(np.float32))
y_test_tensor= torch.from_numpy(y_test.astype(np.float32))

In [ ]:
X_train_tensor.shape


In [ ]:
y_train_tensor.shape

In [ ]:
from torch.utils.data import Dataset,DataLoader

class CustomDataset(Dataset):
  def __init__(self,features,label):
    self.features=features
    self.label=label

  def __len__(self): # Corrected indentation: this method should be at the same level as __init__ and __getitem__
    return len(self.features)

  def __getitem__(self,idx):
    return self.features[idx],self.label[idx]

In [ ]:
train_Dataset=CustomDataset(X_train_tensor,y_train_tensor)
test_Dataset=CustomDataset(X_test_tensor,y_test_tensor)

In [ ]:
train_Dataset[10]

In [ ]:
train_loader=DataLoader(train_Dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_Dataset,batch_size=32,shuffle=True)

In [ ]:
import torch.nn as nn
class SimpleNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear=nn.Linear(num_features,1)
    self.sigmoid=nn.Sigmoid()

  def forward(self,features):
    out= self.linear(features)
    out= self.sigmoid(out)
    return out

In [ ]:
learning_rate=0.1
epochs=25

In [ ]:
model=SimpleNN(X_train_tensor.shape[1])
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)
loss_function=nn.BCELoss()

In [ ]:
for epoch in range(epochs):

  for batch_features,batch_labels in train_loader:
    y_pred=model(batch_features)
    loss=loss_function(y_pred,batch_labels.view(-1, 1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f"Epoch {epoch+1}, Loss:{loss.item()}")

In [ ]:
model.eval()
accuracy_list=[]

In [ ]:
with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    y_pred=model(batch_features)
    y_pred_labels=(y_pred>0.8).float()

    batch_accuracy=(y_pred_labels.view(-1)==batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

overall_accuracy=sum(accuracy_list)/len(accuracy_list)
print(f"Overall Accuracy:{overall_accuracy}")